Define gene subset for GRN Inference

In [1]:
from anndata import AnnData 
import cellrank as cr
import math
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from numpy.linalg import pinv
import ot 
import pandas as pd
import random
import seaborn as sbn
import scipy
import anndata as ad
import scanpy as sc
import scFates as scf
import scipy.stats as stats
from scipy import sparse
from scipy.stats import wasserstein_distance
from scipy.special import gammaln
from scipy.cluster.hierarchy import linkage, to_tree, dendrogram
from scipy.spatial.distance import pdist
import scvelo as scv
from joblib import Parallel, delayed

/Users/olivier_2/carda_venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
# Hyperparameter values
SFT= 12.5 # time scale factor
s=30 # point size for scatter plots
f=10 # stabilization factor for mRNA degradation rates
DRR=10 # degradation rate ratio limit
CC = 20  # cell cycle time

path="/Users/olivier_2/Documents/En_cours/Labo/Manipes/OG3700/"

1. Load the Anndata object

In [3]:
adata = ad.read_h5ad(path+'adata_3700_2.h5ad')

2. Generates the file for computing entropy

In [4]:
df_gene_expression = pd.DataFrame(
    adata.layers['counts'].toarray(), 
    index=adata.obs.index,
    columns=adata.var.index
)

df_gene_expression['time'] = adata.obs['time'].values
times = df_gene_expression['time'].values
gene_columns = [c for c in df_gene_expression.columns if c != 'time']

counts_matrix = df_gene_expression[gene_columns].values.astype(np.float32)

unique_times = np.unique(times)
print(f"Detected time points: {unique_times}")
print(f"Matrix shape: {counts_matrix.shape[0]} cells x {counts_matrix.shape[1]} genes")

Detected time points: [ 0.  12.5 25.  37.5 50. ]
Matrix shape: 4345 cells x 30034 genes


2. Define the KD function

In [5]:
# 2. Optimized calculation function (parallelized)
def compute_pairwise_distances(t1, t2, matrix, times):
    # Get indices of cells belonging to time t1 and t2
    idx1 = np.where(times == t1)[0]
    idx2 = np.where(times == t2)[0]
    
    # Safety check: skip if one of the time points has no cells
    if len(idx1) == 0 or len(idx2) == 0:
        return []
        
    # Extract sub-matrices for the two time points (very fast with NumPy slicing)
    # data1 contains all cells from t1, all genes
    # data2 contains all cells from t2, all genes
    data1 = matrix[idx1, :]
    data2 = matrix[idx2, :]
    
    # Calculate Wasserstein distance for each gene (column) in parallel
    # n_jobs=-1 uses all available CPU cores
    # We iterate over genes (columns) comparing the distribution of cells in t1 vs t2
    dists = Parallel(n_jobs=-1)(
        delayed(wasserstein_distance)(data1[:, g], data2[:, g])
        for g in range(data1.shape[1])
    )
    return dists

3. Compute KD distances for all time point pairs

In [6]:
# 3. Execute calculation for consecutive time points
kanto_results = []
time_pairs = []

print("Starting distance calculations...")
for i in range(len(unique_times) - 1):
    t_start = unique_times[i]
    t_end = unique_times[i+1]
    print(f"  -> Calculating between {t_start} and {t_end}...")
    
    # Compute distances for all genes between these two time points
    dists = compute_pairwise_distances(t_start, t_end, counts_matrix, times)
    
    # Append results to the main list
    kanto_results.extend(dists)
    time_pairs.append((t_start, t_end))


# Optional: Organize results into a DataFrame for easier analysis
results_df = pd.DataFrame({
    'Gene': gene_columns * len(time_pairs), # Repeat gene names for each time transition
    'Distance': kanto_results,
    'Time_Transition': [f"{t1}_to_{t2}" for t1, t2 in time_pairs for _ in range(len(gene_columns))]
})

Starting distance calculations...
  -> Calculating between 0.0 and 12.5...
  -> Calculating between 12.5 and 25.0...
  -> Calculating between 25.0 and 37.5...
  -> Calculating between 37.5 and 50.0...


4. Compute MDE distances for all time point pairs

In [7]:
entropy_matrix = pd.read_csv(path + 'gene_entropy_by_time.csv', index_col=0)

# Calculation of absolute entropy difference (Delta Entropy) per transition
# Rows = Transitions (t1_vs_t2...), Columns = Genes
delta_entropy_list = []
transition_labels = []

timepoints = sorted(entropy_matrix.columns)

for i in range(len(timepoints) - 1):
    t_curr = timepoints[i]
    t_next = timepoints[i+1]
    
    label = f"{t_curr}_vs_{t_next}"
    transition_labels.append(label)
    
    # Retrieve entropy vectors for all genes
    H_curr = entropy_matrix[t_curr].values
    H_next = entropy_matrix[t_next].values
    
    # Calculate the absolute value of the difference
    delta_H = np.abs(H_next - H_curr)
    
    delta_entropy_list.append(delta_H)

# Create the final DataFrame
df_delta_entropy = pd.DataFrame(
    delta_entropy_list,
    index=transition_labels,
    columns=entropy_matrix.index # Gene names
)

# Save results 
df_delta_entropy.to_csv('delta_entropy_by_transition.csv')
print("\nResults saved to 'delta_entropy_by_transition.csv'")


Results saved to 'delta_entropy_by_transition.csv'


6. Select top MDE and KDE genes

In [8]:
TOP_N = 200

def get_top_KDE(group):
    # Sort the group by 'Distance' in descending order and keep the top N
    return group.sort_values(by='Distance', ascending=False).head(TOP_N)

# Apply sorting and selection for each 'Time_Transition'
top_100_by_transition = results_df.groupby('Time_Transition', group_keys=False).apply(get_top_KDE)

# Get the unique list of genes from the combined top groups
liste_union_genes_KDE = top_100_by_transition['Gene'].unique()
unique_liste_union_genes_KDE = sorted(list(liste_union_genes_KDE))

print(f"Number of unique KDE genes in the top {TOP_N} of each transition: {len(liste_union_genes_KDE)}")

Number of unique KDE genes in the top 200 of each transition: 349


In [9]:
# Selection of top most variable genes in MDE
top_ENT_genes_by_transition = {}

# 1. Extract top genes for each transition (each row)
for transition in df_delta_entropy.index:
    # Select the row corresponding to the transition
    row = df_delta_entropy.loc[transition]
    top_genes = row.nlargest(TOP_N).index.tolist()
    top_ENT_genes_by_transition[transition] = top_genes

sets_of_genes = [set(genes) for genes in top_ENT_genes_by_transition.values()]

# union(*sets) merges all sets into one
unique_genes = set().union(*sets_of_genes)
# Intersection of all sets 
# unique_genes = set().intersection(*sets_of_genes)
unique_genes_list_delta_entropy = sorted(list(unique_genes))

print(f"Number of unique MDE genes in the top {TOP_N} of each transition: {len(unique_genes_list_delta_entropy)}")


# Save results
pd.DataFrame(unique_genes_list_delta_entropy, columns=[f'Union_Top_{TOP_N}_Genes']).to_csv(f'union_top_{TOP_N}_genes_all_transitions.csv', 
                                                                                           index=False)
print(f"\nList saved to 'union_top_{TOP_N}_genes_all_transitions.csv'")

Number of unique MDE genes in the top 200 of each transition: 401

List saved to 'union_top_200_genes_all_transitions.csv'


7. Generate matrix CARDAMOM-ready

In [11]:
# Get the original count matrix
adata_2=ad.read_h5ad('merged_126_226.h5ad')

# Tranfer labels
adata_2.obs['cell_type'] = adata.obs['cell_type']
adata_2.obs['time'] = adata.obs['time']

# Reduce adata to common genes
common_genes = set(liste_union_genes_KDE).intersection(set(unique_genes_list_delta_entropy))
print(len(common_genes))
adata = adata_2[:, adata_2.var_names.isin(list(common_genes))].copy()

# Save gene list
common_genes_list = adata.var_names
common_genes_list = list(common_genes_list)
common_genes_list.sort()
common_genes_list.insert(0, 'stimulus')  
df_common = pd.DataFrame({'genenames': common_genes_list})
df_common.to_csv('common_genes.csv', index=False)


77


8. Add degradation rates

In [12]:
# Adding half-lives to the dataset
path_HL="/Users/olivier_2/Documents/En_cours/Labo/Manipes/OG3691/"
DV = pd.read_csv(path_HL+'merge_table.csv')

cols_to_keep = ["gene_symbol", 
    "rna_half_life", 
    "prot_half_life"
]

available_cols = [c for c in cols_to_keep if c in DV.columns]
DV_short = DV[available_cols].copy()

if 'prot_half_life' in DV_short.columns:
    # Force numeric conversion (in case strings are present)
    DV_short['prot_half_life'] = pd.to_numeric(DV_short['prot_half_life'], errors='coerce')
    
    # Replace NaN with CC
    DV_short['prot_half_life'] = DV_short['prot_half_life'].fillna(CC)
    
    # Cap values at CC
    DV_short['prot_half_life'] = DV_short['prot_half_life'].apply(lambda x: min(x, CC))

# Remove duplicates
DV_short_unique = DV_short.drop_duplicates()

dr = []
# Retrieve the list of genes
for gene in adata.var_names:
    if gene in DV_short_unique['gene_symbol'].values:
        # Retrieve the corresponding row
        row = DV_short_unique[DV_short_unique['gene_symbol'] == gene].iloc[0]
        hl_rna = float(row['rna_half_life'])
        hl_prot = float(row['prot_half_life'])
        dr.append([hl_rna, hl_prot])
    else:
        # Use global average if the gene is not found
        mean_rna = pd.to_numeric(DV_short_unique['rna_half_life'], errors='coerce').mean()
        mean_prot = pd.to_numeric(DV_short_unique['prot_half_life'], errors='coerce').mean()
        dr.append([mean_rna, mean_prot])

# Convert to numpy array for vectorized calculations
dr = np.array(dr)

# Transform half-lives into degradation rates
d_rate = np.log(2) / dr

# To stabilize the mRNAs by a factor f
d_rate[:, 0] = d_rate[:, 0] / f

# Let's limit the degradation rate ratio to DRR
ratio = d_rate[:, 0] / d_rate[:, 1]
mask = ratio > DRR
d_rate[:, 0] = DRR * d_rate[:, 1]

adata.var['d0'] = d_rate[:, 0]
adata.var['d1'] = d_rate[:, 1]

9. Save

In [13]:
adata.write_h5ad('adata_3700_3.h5ad')